# Analyse probing results

Loads every `probing/experiment.py` run under `OUTPUTS_ROOTS` (default
`outputs_probing_colon/`, see `configs/probing_colon.yaml`'s `output_dir`) and
plots them, with a tickable list of which model/correction/token
("embeddings") combinations to include.

## Output layout this reads (see `experiment.py`'s `main()` / `run_split_experiment`)

```
{OUTPUTS_ROOT}/{model_name}/{correction_name}/{mapping}/{matching}/{embeddings}/{split_label}/
    split_informations.yaml
    probes_summary.yaml              <- one entry per probe: {skipped, best_lambda,
                                         val_selection_metric, test_selection_metric}
                                         or {skipped: true, reason: ...}
    {probe_name}/
        results_summary.yaml         <- full per-probe detail: task_type, target_columns,
                                         n_train/n_val/n_test, best_val_metrics, best_test_metrics
        lambda_sweep_results.csv
        per_cell_results.csv
```

`embeddings` is the path segment that encodes **which token(s)** a run used
(`data.embeddings_datasets` -- e.g. `cls`, or the sorted-and-joined corner keys,
or `default` if that config list was left empty, meaning "average every
`embeddings_*` key found in the h5 file"). **This is only meaningful if you gave
different runs of the same model/correction a distinct `embeddings_datasets`
config** -- it's recorded per run, so as long as you did that, this notebook's
"tokens" picker is exactly this segment.

`selection_metric` is R² (`r2_mean`) for every probe except `orientation`,
which uses `-mean_angular_error_deg` instead (circular, mod-180 -- see
`probes/cell_level.py::OrientationProbe.eval_metrics`); this notebook plots
orientation's raw degrees separately too, since R²/degrees aren't comparable
on the same axis.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))


def _load_dotenv(path: Path) -> None:
    """Minimal `.env` loader (KEY=value per line) -- avoids an extra dependency.
    Mirrors how every other entry point in this repo expects env vars to be set
    (see src/python/code_configs/paths.py)."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key, value = key.strip(), value.strip()
        if value:
            os.environ.setdefault(key, value)


_load_dotenv(REPO_ROOT / ".env")

In [ ]:
import math

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    _HAVE_WIDGETS = True
except ImportError:
    _HAVE_WIDGETS = False
    print("ipywidgets is not installed (`pip install ipywidgets`) -- falling back to the plain "
          "SELECTED_CONDITIONS list in the picker cell below; install it for the tickable checkbox UI.")

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## Config

Point `OUTPUTS_ROOTS` at one or more roots holding the `{model_name}/...` tree
(a local copy of `outputs_probing_colon/`, or wherever `configs/probing_colon.yaml`'s
`output_dir` was set to). `MAPPING` / `MATCHING` / `CORRECTION_NAME` only need to
be set if a root mixes more than one value of that path segment -- leave `None`
to auto-detect (a warning prints, and nothing gets filtered out, if more than
one value is found). `correction_name` is *not* forced by default the same way
-- it's folded into the tickable "condition" label below alongside model/token,
since (per `probing_colon.yaml`'s own comment) it's a real axis people compare
(e.g. `Base` vs. a batch-correction method), not just a fixed experiment-design
choice.

In [ ]:
OUTPUTS_ROOTS = [Path("../outputs_probing_colon")]

MAPPING = None    # e.g. "simplified_broad" -- force one value if a root mixes several
MATCHING = None   # e.g. "matching" or "all_cells"

## Loading

Two passes, mirroring the two YAML files `experiment.py` writes:

1. `load_probe_summaries` walks every `probes_summary.yaml` (one per split --
   cheap, and the only place a *skipped* probe is recorded at all, with its
   skip reason).
2. `load_probe_details` enriches each non-skipped row with its probe's own
   `results_summary.yaml` (`n_train`/`n_val`/`n_test` and the full
   `best_val_metrics`/`best_test_metrics` dict, flattened with a `val_`/`test_`
   prefix -- `r2_<col>`, `r2_mean`, and, for `orientation` only,
   `r2_orientation_cos2`/`r2_orientation_sin2`/`mean_angular_error_deg`).

In [ ]:
SCHEMA = ("model_name", "correction_name", "mapping", "matching", "embeddings")


def _flatten(prefix: str, d: dict) -> dict:
    return {f"{prefix}_{k}": v for k, v in d.items()}


def _split_idx(split_label: str) -> int | None:
    return None if split_label == "same_wsi_split" else int(split_label.removeprefix("split_"))


def load_probe_summaries(root: Path) -> pd.DataFrame:
    """One row per (run, probe) from every probes_summary.yaml under root --
    includes skipped probes (skipped=True, reason=..., no metrics)."""
    rows: list[dict] = []
    for summary_path in sorted(root.rglob("probes_summary.yaml")):
        parts = summary_path.relative_to(root).parts[:-1]  # drop the filename itself
        if len(parts) != len(SCHEMA) + 1:  # + split_label
            print(f"[skip] unexpected path depth ({len(parts)} != {len(SCHEMA) + 1}): {summary_path}")
            continue
        meta = dict(zip(SCHEMA, parts[:-1]))
        split_label = parts[-1]

        with open(summary_path) as f:
            probes = yaml.safe_load(f) or {}

        for probe_name, entry in probes.items():
            rows.append({
                **meta, "split_label": split_label, "split_idx": _split_idx(split_label),
                "is_same_wsi": split_label == "same_wsi_split",
                "probe_dir": summary_path.parent / probe_name.replace("@", "_"),
                "probe": probe_name, **(entry or {}),
            })
    return pd.DataFrame(rows)


def load_probe_details(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich non-skipped rows with their probe's own results_summary.yaml."""
    extra_rows = []
    for _, row in df.iterrows():
        extra: dict = {}
        if not row.get("skipped", True):
            path = Path(row["probe_dir"]) / "results_summary.yaml"
            if path.exists():
                with open(path) as f:
                    summary = yaml.safe_load(f)
                extra = {
                    "task_type": summary.get("task_type"), "target_columns": summary.get("target_columns"),
                    "n_train": summary.get("n_train"), "n_val": summary.get("n_val"), "n_test": summary.get("n_test"),
                    **_flatten("val", summary.get("best_val_metrics") or {}),
                    **_flatten("test", summary.get("best_test_metrics") or {}),
                }
        extra_rows.append(extra)
    extra_df = pd.DataFrame(extra_rows, index=df.index)
    return pd.concat([df, extra_df], axis=1)


def _pick(df: pd.DataFrame, col: str, forced):
    values = df[col].unique().tolist()
    if forced is not None:
        return forced
    if len(values) > 1:
        print(f"[warning] multiple {col!r} values found {values} -- "
              f"set {col.upper()} above to filter to one; keeping all of them for now")
        return None
    return values[0]


results_df = pd.concat([load_probe_summaries(root) for root in OUTPUTS_ROOTS], ignore_index=True)
assert not results_df.empty, f"no probes_summary.yaml found under {[str(r.resolve()) for r in OUTPUTS_ROOTS]}"
results_df = load_probe_details(results_df)

for col, forced in (("mapping", MAPPING), ("matching", MATCHING)):
    picked = _pick(results_df, col, forced)
    if picked is not None:
        results_df = results_df[results_df[col] == picked]

results_df["condition"] = (
    results_df["model_name"] + " | " + results_df["correction_name"] + " | " + results_df["embeddings"]
)

print(f"Loaded {len(results_df)} (run x probe) rows -- "
      f"{results_df['condition'].nunique()} condition(s): {sorted(results_df['condition'].unique())}; "
      f"probes: {sorted(results_df['probe'].unique())}")
results_df.head()

## Plotting helpers

`render(selected_conditions)` redraws everything below from `results_df`
filtered to `selected_conditions` -- the picker cell further down just calls
this on every tick. Faceted per probe throughout, since `selection_metric`'s
scale/sign differs by probe type (R² vs. `-mean_angular_error_deg`) -- each
probe gets its own subplot with its own y-axis.

In [ ]:
def _metric_grid(n: int, figsize_per=(5.5, 4)):
    ncols = 2 if n > 1 else 1
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per[0] * ncols, figsize_per[1] * nrows), squeeze=False)
    axes = axes.flatten()
    for ax in axes[n:]:
        ax.axis("off")
    return fig, axes


def _grouped_bar(ax, pivot_mean: pd.DataFrame, pivot_std: pd.DataFrame, ylabel: str):
    """pivot_*: index=condition, columns=hue (e.g. ['val', 'test'] or ['LWO mean', 'same_wsi_split'])."""
    conditions = pivot_mean.index.tolist()
    hues = pivot_mean.columns.tolist()
    x = np.arange(len(conditions))
    width = 0.8 / max(len(hues), 1)
    for i, hue in enumerate(hues):
        means = pivot_mean[hue].values
        stds = pivot_std[hue].values if pivot_std is not None and hue in pivot_std else np.zeros_like(means)
        ax.bar(x + i * width - 0.4 + width / 2, means, width=width, yerr=stds, capsize=3, label=str(hue))
    ax.set_xticks(x)
    ax.set_xticklabels(conditions, rotation=25, ha="right", fontsize=7)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7)
    ax.grid(axis="y", alpha=0.3)
    ax.axhline(0, color="black", linewidth=0.6)


def plot_val_vs_test_by_probe(df: pd.DataFrame) -> None:
    """LWO-only: mean +/- std selection_metric across splits, val vs test, one subplot per probe."""
    loso = df[~df["is_same_wsi"]]
    probes = sorted(loso["probe"].unique())
    fig, axes = _metric_grid(len(probes))
    for ax, probe in zip(axes, probes):
        sub = loso[loso["probe"] == probe]
        mean_pivot = sub.pivot_table(index="condition", columns=[], values=["val_selection_metric", "test_selection_metric"], aggfunc="mean")
        std_pivot = sub.pivot_table(index="condition", columns=[], values=["val_selection_metric", "test_selection_metric"], aggfunc="std")
        mean_pivot = mean_pivot.rename(columns={"val_selection_metric": "val", "test_selection_metric": "test"})
        std_pivot = std_pivot.rename(columns={"val_selection_metric": "val", "test_selection_metric": "test"})
        _grouped_bar(ax, mean_pivot, std_pivot, "selection_metric")
        ax.set_title(probe, fontsize=9)
    fig.suptitle("selection_metric: mean +/- std across LWO splits (val vs. test)")
    fig.tight_layout()
    plt.show()


def plot_per_split_lines(df: pd.DataFrame, value_col: str = "test_selection_metric") -> None:
    """LWO-only: value_col per split_idx, one line per condition, one subplot per probe --
    the fold-to-fold stability check (is a probe's result consistent across held-out WSIs,
    or is one fold an outlier?)."""
    loso = df[~df["is_same_wsi"]]
    probes = sorted(loso["probe"].unique())
    fig, axes = _metric_grid(len(probes))
    for ax, probe in zip(axes, probes):
        sub = loso[loso["probe"] == probe]
        pivot = sub.pivot_table(index="split_idx", columns="condition", values=value_col)
        for condition in pivot.columns:
            ax.plot(pivot.index, pivot[condition], marker="o", label=condition, markersize=4)
        ax.set_title(probe, fontsize=9)
        ax.set_xlabel("split_idx")
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=6, loc="best")
    fig.suptitle(f"{value_col} per LWO split")
    fig.tight_layout()
    plt.show()


def plot_lwo_vs_same_wsi(df: pd.DataFrame, value_col: str = "test_selection_metric") -> None:
    """LWO mean vs. same_wsi_split (pooled 70/15/15 across all WSIs) for value_col,
    one subplot per probe -- the WSI-transfer check: a big gap here means the target
    only "works" when train/test share a slide (see geometry.py's orientation_deg,
    defined in each WSI's own absolute pixel frame)."""
    loso = df[~df["is_same_wsi"]]
    same_wsi = df[df["is_same_wsi"]]
    probes = sorted(df["probe"].unique())
    fig, axes = _metric_grid(len(probes))
    for ax, probe in zip(axes, probes):
        lwo_mean = loso[loso["probe"] == probe].groupby("condition")[value_col].mean().rename("LWO mean")
        sw = same_wsi[same_wsi["probe"] == probe].groupby("condition")[value_col].mean().rename("same_wsi_split")
        pivot_mean = pd.concat([lwo_mean, sw.reindex(lwo_mean.index)], axis=1)
        _grouped_bar(ax, pivot_mean, None, value_col)
        ax.set_title(probe, fontsize=9)
    fig.suptitle(f"{value_col}: Leave-WSI-Out mean vs. same-WSI upper bound")
    fig.tight_layout()
    plt.show()


def plot_orientation_degrees(df: pd.DataFrame) -> None:
    """Orientation's own selection_metric is -mean_angular_error_deg -- not comparable
    to other probes' R^2 on the same axis, so it gets its own degrees-native view, with
    the two reference lines that make the number interpretable: 45 deg = chance (two
    independent mod-180 angles), 90 deg = worst case (systematically perpendicular)."""
    orient = df[df["probe"] == "orientation"]
    if orient.empty:
        return

    loso = orient[~orient["is_same_wsi"]]
    same_wsi = orient[orient["is_same_wsi"]]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    pivot = loso.pivot_table(index="split_idx", columns="condition", values="test_mean_angular_error_deg")
    for condition in pivot.columns:
        axes[0].plot(pivot.index, pivot[condition], marker="o", label=condition, markersize=4)
    axes[0].axhline(45, color="gray", linestyle="--", linewidth=1, label="45 deg = chance")
    axes[0].axhline(90, color="red", linestyle=":", linewidth=1, label="90 deg = worst case")
    axes[0].set_xlabel("split_idx")
    axes[0].set_ylabel("test mean angular error (deg)")
    axes[0].set_title("orientation: per-split test error")
    axes[0].legend(fontsize=6)
    axes[0].grid(alpha=0.3)

    lwo_mean = loso.groupby("condition")["test_mean_angular_error_deg"].mean().rename("LWO mean")
    sw = same_wsi.groupby("condition")["test_mean_angular_error_deg"].mean().rename("same_wsi_split")
    pivot_mean = pd.concat([lwo_mean, sw.reindex(lwo_mean.index)], axis=1)
    _grouped_bar(axes[1], pivot_mean, None, "test mean angular error (deg)")
    axes[1].axhline(45, color="gray", linestyle="--", linewidth=1)
    axes[1].axhline(90, color="red", linestyle=":", linewidth=1)
    axes[1].set_title("orientation: LWO mean vs. same-WSI")

    fig.suptitle("orientation probe, in its native units (deg) -- not comparable to other probes' R^2")
    fig.tight_layout()
    plt.show()


def render(selected_conditions: list[str]) -> None:
    df = results_df[results_df["condition"].isin(selected_conditions)]
    if df.empty:
        print("No rows match the current selection.")
        return

    skipped = df[df["skipped"] == True]  # noqa: E712 -- explicit bool compare reads clearer here
    if not skipped.empty:
        print(f"{len(skipped)} skipped (probe, run) row(s) in this selection:")
        print(skipped[["condition", "split_label", "probe", "reason"]].to_string(index=False))

    fit = df[df["skipped"] == False]  # noqa: E712
    if fit.empty:
        print("Every selected row was skipped -- nothing to plot.")
        return

    plot_val_vs_test_by_probe(fit)
    plot_per_split_lines(fit, "test_selection_metric")
    plot_lwo_vs_same_wsi(fit, "test_selection_metric")
    plot_orientation_degrees(fit)

    print(f"\n{len(fit)} fitted (run x probe) rows for the current selection:")
    display_cols = ["condition", "split_label", "probe", "best_lambda", "n_train", "n_val", "n_test",
                     "val_selection_metric", "test_selection_metric"]
    print(fit[display_cols].sort_values(["condition", "probe", "split_label"]).to_string(index=False))

## Pick which models / corrections / tokens to plot

One checkbox per `condition` (`model_name | correction_name | embeddings`) --
tick/untick and the plots above redraw automatically. Falls back to a plain
`SELECTED_CONDITIONS` list (edit it directly, then re-run this cell) if
`ipywidgets` isn't installed.

In [ ]:
ALL_CONDITIONS = sorted(results_df["condition"].unique())

if _HAVE_WIDGETS:
    _checkboxes = [widgets.Checkbox(value=True, description=c, indent=False,
                                     layout=widgets.Layout(width="max-content"))
                   for c in ALL_CONDITIONS]
    _output_area = widgets.Output()

    def _on_change(_change):
        selected = [cb.description for cb in _checkboxes if cb.value]
        with _output_area:
            clear_output(wait=True)
            render(selected)

    for _cb in _checkboxes:
        _cb.observe(_on_change, names="value")

    _select_all = widgets.Button(description="Select all", layout=widgets.Layout(width="100px"))
    _select_none = widgets.Button(description="Select none", layout=widgets.Layout(width="100px"))

    def _set_all(_btn, value: bool):
        for cb in _checkboxes:
            cb.unobserve(_on_change, names="value")
        for cb in _checkboxes:
            cb.value = value
        for cb in _checkboxes:
            cb.observe(_on_change, names="value")
        _on_change(None)

    _select_all.on_click(lambda _btn: _set_all(_btn, True))
    _select_none.on_click(lambda _btn: _set_all(_btn, False))

    picker = widgets.VBox([
        widgets.HBox([widgets.HTML("<b>Models / corrections / tokens to plot:</b>"), _select_all, _select_none]),
        widgets.VBox(_checkboxes, layout=widgets.Layout(max_height="240px", overflow="auto", border="1px solid #ccc")),
    ])
    display(picker, _output_area)
    _on_change(None)
else:
    SELECTED_CONDITIONS = list(ALL_CONDITIONS)  # edit this list, then re-run this cell
    render(SELECTED_CONDITIONS)